# Preprocesamiento del dataset PRO-ACT
## Objetivo: construir el dataset final para entrenar los modelos de predicción de progresión en ELA

## 1. Carga de librerías y datos

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("✅ Librerías cargadas")

✅ Librerías cargadas


### Carga de datos

Se definen las rutas de entrada y salida y se cargan los seis archivos CSV del dataset PRO-ACT.

In [4]:
ruta_raw  = r"C:\Users\Paula\SAD_ELA\csv_proact/"
ruta_out  = r"C:\Users\Paula\SAD_ELA\datos_procesados/"

alsfrs   = pd.read_csv(ruta_raw + "F_PROACT_ALSFRS.csv")
demo     = pd.read_csv(ruta_raw + "F_PROACT_DEMOGRAPHICS.csv")
historia = pd.read_csv(ruta_raw + "F_PROACT_ALSHISTORY.csv")
fvc      = pd.read_csv(ruta_raw + "F_PROACT_FVC.csv")
vitals   = pd.read_csv(ruta_raw + "F_PROACT_VITALSIGNS.csv", low_memory=False)
labs     = pd.read_csv(ruta_raw + "F_PROACT_LABS.csv")

print("✅ Archivos cargados correctamente")

✅ Archivos cargados correctamente


## 2. Extracción de variables del ALSFRS

El archivo ALSFRS contiene las mediciones longitudinales de la escala ALSFRS-R.
De este archivo extraemos dos tipos de información:
- **Tasa de progresión**: pendiente de la regresión lineal de cada paciente,
  usada para crear la variable objetivo
- **Variables basales**: puntuaciones de la primera visita de cada paciente,
  usadas como predictores del modelo

In [5]:
# Selección de columnas relevantes
# Se descartan columnas con más del 50% de nulos (ver 01_exploracion.ipynb):
# Q5b_Cutting_with_Gastrostomy (92%), Q10_Respiratory (55%),
# ALSFRS_Total (55%), Mode_of_Administration (90%), ALSFRS_Responded_By (97%)

cols_alsfrs = [
    'subject_id', 'ALSFRS_Delta',
    'Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing',
    'Q4_Handwriting', 'Q5a_Cutting_without_Gastrostomy',
    'Q6_Dressing_and_Hygiene', 'Q7_Turning_in_Bed',
    'Q8_Walking', 'Q9_Climbing_Stairs',
    'R_1_Dyspnea', 'R_2_Orthopnea', 'R_3_Respiratory_Insufficiency',
    'ALSFRS_R_Total'
]

alsfrs_clean = alsfrs[cols_alsfrs].copy()
print("Shape ALSFRS seleccionado:", alsfrs_clean.shape)

Shape ALSFRS seleccionado: (81229, 15)


### 2.1 Cálculo de la tasa de progresión por paciente

Para cada paciente calculamos la pendiente de la recta que mejor ajusta 
sus mediciones ALSFRS-R a lo largo del tiempo mediante regresión lineal simple.
Esta pendiente representa los puntos perdidos por mes — cuanto más negativa, 
más rápida es la progresión.

Posteriormente filtramos valores extremos (fuera del rango -10 a 2) que 
corresponden a errores de medición o pacientes con muy pocas visitas, y 
clasificamos a cada paciente con un umbral de -1 punto/mes respaldado 
por Atassi et al. (2014):
- **Progresión rápida (1)**: tasa < -1 punto/mes
- **Progresión lenta (0)**: tasa ≥ -1 punto/mes

In [6]:
def calcular_tasa(grupo):
    g = grupo.dropna(subset=['ALSFRS_R_Total', 'ALSFRS_Delta'])
    if len(g) < 2:
        return np.nan
    tiempo_meses = g['ALSFRS_Delta'] / 30.44
    slope, _, _, _, _ = stats.linregress(tiempo_meses, g['ALSFRS_R_Total'])
    return slope

tasas = alsfrs_clean.groupby('subject_id').apply(
    calcular_tasa, include_groups=False
)
tasas.name = 'tasa_progresion'
df_tasas = tasas.reset_index()

# Filtrar extremos y crear variable objetivo
df_tasas = df_tasas[df_tasas['tasa_progresion'].between(-10, 2)]
df_tasas['progresion_rapida'] = (df_tasas['tasa_progresion'] < -1).astype(int)
df_tasas['etiqueta'] = df_tasas['progresion_rapida'].map({1: 'Rápida', 0: 'Lenta'})

n_total  = len(df_tasas)
n_rapida = df_tasas['progresion_rapida'].sum()
n_lenta  = n_total - n_rapida

print(f"Total pacientes: {n_total}")
print(f"Progresión rápida (1): {n_rapida} ({n_rapida/n_total*100:.1f}%)")
print(f"Progresión lenta  (0): {n_lenta} ({n_lenta/n_total*100:.1f}%)")

Total pacientes: 5689
Progresión rápida (1): 2574 (45.2%)
Progresión lenta  (0): 3115 (54.8%)


### 2.2 Extracción de variables basales del ALSFRS

Tomamos la primera medición de cada paciente como estado basal, 
que refleja su situación al inicio del seguimiento.
Calculamos además la puntuación agregada por dominio clínico:
bulbar, motor fino, motor grueso y respiratorio.

In [7]:
alsfrs_sorted = alsfrs_clean.sort_values(['subject_id', 'ALSFRS_Delta'])
alsfrs_basal  = alsfrs_sorted.groupby('subject_id').first().reset_index()

alsfrs_basal['basal_bulbar'] = (
    alsfrs_basal['Q1_Speech'] +
    alsfrs_basal['Q2_Salivation'] +
    alsfrs_basal['Q3_Swallowing']
)
alsfrs_basal['basal_motor_fino'] = (
    alsfrs_basal['Q4_Handwriting'] +
    alsfrs_basal['Q5a_Cutting_without_Gastrostomy'] +
    alsfrs_basal['Q6_Dressing_and_Hygiene']
)
alsfrs_basal['basal_motor_grueso'] = (
    alsfrs_basal['Q7_Turning_in_Bed'] +
    alsfrs_basal['Q8_Walking'] +
    alsfrs_basal['Q9_Climbing_Stairs']
)
alsfrs_basal['basal_respiratorio'] = (
    alsfrs_basal['R_1_Dyspnea'] +
    alsfrs_basal['R_2_Orthopnea'] +
    alsfrs_basal['R_3_Respiratory_Insufficiency']
)

alsfrs_basal = alsfrs_basal[[
    'subject_id', 'ALSFRS_R_Total',
    'basal_bulbar', 'basal_motor_fino',
    'basal_motor_grueso', 'basal_respiratorio'
]].rename(columns={'ALSFRS_R_Total': 'alsfrs_basal_total'})

print("Shape:", alsfrs_basal.shape)
print("\nNulos por columna:")
print(alsfrs_basal.isnull().sum())

Shape: (9149, 6)

Nulos por columna:
subject_id               0
alsfrs_basal_total    3102
basal_bulbar           330
basal_motor_fino       440
basal_motor_grueso     330
basal_respiratorio    3432
dtype: int64


### Tratamiento de nulos en variables basales ALSFRS

Los nulos en `alsfrs_basal_total`(34%) y `basal_respiratorio` (37.5%) se deben 
a que la versión Revised de la escala (ALSFRS-R) se incorporó más tarde al 
protocolo PRO-ACT — los pacientes más antiguos no tienen estas mediciones.

En todos los casos aplicamos **imputación con la mediana** porque:
- Es robusta frente a valores extremos, algo importante en datos clínicos reales
- Nos permite conservar todos los pacientes en lugar de eliminar filas,
  lo cual sería muy costoso dado el alto porcentaje de nulos

In [8]:
cols_imputar = [
    'alsfrs_basal_total', 'basal_bulbar',
    'basal_motor_fino', 'basal_motor_grueso', 'basal_respiratorio'
]

for col in cols_imputar:
    mediana = alsfrs_basal[col].median()
    alsfrs_basal[col] = alsfrs_basal[col].fillna(mediana)
    print(f"{col} → mediana: {mediana:.1f}")

print("\n✅ Nulos tras imputación:")
print(alsfrs_basal.isnull().sum())

alsfrs_basal_total → mediana: 38.0
basal_bulbar → mediana: 11.0
basal_motor_fino → mediana: 9.0
basal_motor_grueso → mediana: 8.0
basal_respiratorio → mediana: 12.0

✅ Nulos tras imputación:
subject_id            0
alsfrs_basal_total    0
basal_bulbar          0
basal_motor_fino      0
basal_motor_grueso    0
basal_respiratorio    0
dtype: int64


## 3. Extracción de variables demográficas

De Demographics extraemos edad y sexo.
Estas variables son predictores básicos pero importantes; la edad y el sexo
son factores de riesgo conocidos en ELA.

In [9]:
# Selección de columnas relevantes
# Se descartan: Date_of_Birth (85% nulos), Ethnicity (66%), 
# todas las Race_* (>96% nulos) y Demographics_Delta (>92% de valores nulos)

cols_demo = ['subject_id', 'Age', 'Sex']
demo_clean = demo[cols_demo].copy()

print("Shape Demographics seleccionado:", demo_clean.shape)
print("\nNulos por columna:")
print(demo_clean.isnull().sum())

Shape Demographics seleccionado: (13115, 3)

Nulos por columna:
subject_id       0
Age           3021
Sex              6
dtype: int64


### Tratamiento de nulos en Demographics

- `Age` (23% nulos) → **imputación con la mediana** para conservar los pacientes
- `Sex` (0% nulos relevantes) → los 6 casos restantes se eliminarán 
  automáticamente al unir todos los archivos al final

In [10]:
# Imputar Age con la mediana
mediana_edad = demo_clean['Age'].median()
demo_clean['Age'] = demo_clean['Age'].fillna(mediana_edad)

# Convertir Sex a numérico (los modelos solo entienden números)
demo_clean['Sex'] = demo_clean['Sex'].map({'Male': 1, 'Female': 0})

print(f"Mediana de edad usada: {mediana_edad:.1f} años")
print("\n✅ Nulos tras tratamiento:")
print(demo_clean.isnull().sum())
print("\nShape final:", demo_clean.shape)

Mediana de edad usada: 58.0 años

✅ Nulos tras tratamiento:
subject_id    0
Age           0
Sex           6
dtype: int64

Shape final: (13115, 3)


## 4. Extracción de variables de ALSHISTORY

De este archivo extraemos el sitio de inicio de la enfermedad (`Site_of_Onset`),
uno de los predictores clínicos más importantes en ELA. Los pacientes con inicio
bulbar tienen peor pronóstico que los de inicio espinal o en extremidades.

Se descarta el resto de columnas por tener más del 60% de nulos o ser 
redundantes con otras variables ya extraídas.

In [11]:
# Selección de columnas relevantes
historia_clean = historia[['subject_id', 'Site_of_Onset']].copy()

print("Shape ALSHISTORY seleccionado:", historia_clean.shape)

ids_con_valor = set(historia_clean.dropna(subset=['Site_of_Onset'])['subject_id'])
ids_totales   = set(historia_clean['subject_id'])
ids_nulos_reales = ids_totales - ids_con_valor

print(f"IDs totales: {len(ids_totales)}")
print(f"IDs nulos reales: {len(ids_nulos_reales)}")

print("\nValores únicos de Site_of_Onset:")
print(historia_clean['Site_of_Onset'].value_counts())

Shape ALSHISTORY seleccionado: (14321, 2)
IDs totales: 11656
IDs nulos reales: 5939

Valores únicos de Site_of_Onset:
Site_of_Onset
Onset: Limb               4077
Onset: Bulbar             1150
Onset: Other               362
Onset: Spine                95
Onset: Limb and Bulbar      33
Name: count, dtype: int64


### Tratamiento de nulos en ALSHISTORY

`Site_of_Onset` tiene un 51% de nulos (5.939 pacientes). 

En lugar de imputar con la moda, creamos una categoría explícita **Unknown** para los valores ausentes.

Imputar con la moda distorsionaría la distribución clínica real; los 5.939
nulos se convertirían artificialmente en "Onset: Limb", inflando esa categoría
de 4.077 a 10.016 pacientes. Conservar el desconocimiento como categoría propia
es más honesto y puede ser informativo para el modelo.

Posteriormente aplicamos **one-hot encoding** creando una columna binaria por 
cada categoría: Limb, Bulbar, Spine, Limb and Bulbar, Other y Unknown.

In [12]:
# Deduplicar: un registro por paciente, priorizando filas CON valor
historia_clean = (
    historia_clean
    .sort_values('Site_of_Onset', na_position='last')
    .drop_duplicates(subset='subject_id', keep='first')
    .reset_index(drop=True)
)

print(f"Shape tras deduplicación: {historia_clean.shape}")
print(f"Nulos reales: {historia_clean['Site_of_Onset'].isna().sum()} "
      f"({historia_clean['Site_of_Onset'].isna().mean()*100:.1f}%)")

Shape tras deduplicación: (11656, 2)
Nulos reales: 5939 (51.0%)


### Codificación one-hot de `Site_of_Onset`

Rellenamos los nulos con la categoría `Unknown` y aplicamos **one-hot encoding** creando una columna binaria por cada categoría: Limb, Bulbar, Spine, Limb and Bulbar, Other y Unknown.

In [13]:

historia_clean['Site_of_Onset'] = historia_clean['Site_of_Onset'].fillna('Unknown')

# One-hot encoding para todas las categorías incluyendo Unknown
historia_clean['onset_limb']          = (historia_clean['Site_of_Onset'] == 'Onset: Limb').astype(int)
historia_clean['onset_bulbar']        = (historia_clean['Site_of_Onset'] == 'Onset: Bulbar').astype(int)
historia_clean['onset_spine']         = (historia_clean['Site_of_Onset'] == 'Onset: Spine').astype(int)
historia_clean['onset_limb_bulbar']   = (historia_clean['Site_of_Onset'] == 'Onset: Limb and Bulbar').astype(int)
historia_clean['onset_other']         = (historia_clean['Site_of_Onset'] == 'Onset: Other').astype(int)
historia_clean['onset_unknown']       = (historia_clean['Site_of_Onset'] == 'Unknown').astype(int)

historia_clean = historia_clean[[
    'subject_id', 'onset_limb', 'onset_bulbar',
    'onset_spine', 'onset_limb_bulbar', 'onset_other', 'onset_unknown'
]]

print("\n✅ Nulos tras tratamiento:")
print(historia_clean.isnull().sum())
print("\nShape final:", historia_clean.shape)
print("\nDistribución de categorías:")
print(historia_clean[['onset_limb','onset_bulbar','onset_spine',
                       'onset_limb_bulbar','onset_other','onset_unknown']].sum())


✅ Nulos tras tratamiento:
subject_id           0
onset_limb           0
onset_bulbar         0
onset_spine          0
onset_limb_bulbar    0
onset_other          0
onset_unknown        0
dtype: int64

Shape final: (11656, 7)

Distribución de categorías:
onset_limb           4077
onset_bulbar         1150
onset_spine            95
onset_limb_bulbar      33
onset_other           362
onset_unknown        5939
dtype: int64


## 5. Extracción de variables de FVC

La Capacidad Vital Forzada (FVC) mide la función respiratoria del paciente.
Es un predictor importante de progresión en ELA ya que el deterioro respiratorio
es la principal causa de muerte.

De este archivo extraemos `Subject_Liters_Trial_1`, la única columna con datos
suficientes. El resto de columnas tienen más del 60% de nulos y se descartan.

> **Nota sobre los niveles de análisis de nulos**: el archivo FVC es longitudinal,
> es decir, cada paciente tiene múltiples filas (una por visita). Por eso los nulos
> se analizan en dos niveles:
> - **A nivel de fila** (sobre las ~49.110 filas del archivo): el `isnull().sum()` de
>   abajo muestra que el **6.7%** de las filas no tienen valor en `Subject_Liters_Trial_1`.
> - **A nivel de paciente** (sobre IDs únicos): lo relevante es cuántos pacientes
>   no tienen **ninguna** fila con valor — esos son los *nulos reales* que se cuantifican
>   al final de esta celda y que habrá que imputar.

In [14]:
# Selección de columnas relevantes
fvc_clean = fvc[['subject_id', 'Forced_Vital_Capacity_Delta', 
                  'Subject_Liters_Trial_1']].copy()

print("Shape FVC seleccionado:", fvc_clean.shape)
print("\nIDs únicos:", fvc_clean['subject_id'].nunique())

print("\nNulos por columna (nivel fila):")
print(fvc_clean.isnull().sum())

ids_con_valor_Subject_Liters_Trial_1 = set(fvc_clean.dropna(subset=['Subject_Liters_Trial_1'])['subject_id'])
ids_totales   = set(fvc_clean['subject_id'])
ids_nulos_reales_Subject_Liters_Trial_1 = ids_totales - ids_con_valor_Subject_Liters_Trial_1

print(f"\nNulos a nivel de paciente (IDs únicos):")
print(f"  IDs totales:                         {len(ids_totales)}")
print(f"  IDs con al menos un valor:           {len(ids_con_valor_Subject_Liters_Trial_1)}")
print(f"  IDs sin ningún valor (nulos reales): {len(ids_nulos_reales_Subject_Liters_Trial_1)}")
print(f"  % nulos reales (sobre IDs únicos):   {len(ids_nulos_reales_Subject_Liters_Trial_1)/len(ids_totales)*100:.1f}%")

Shape FVC seleccionado: (49110, 3)

IDs únicos: 9090

Nulos por columna (nivel fila):
subject_id                        0
Forced_Vital_Capacity_Delta    2558
Subject_Liters_Trial_1         3294
dtype: int64

Nulos a nivel de paciente (IDs únicos):
  IDs totales:                         9090
  IDs con al menos un valor:           7711
  IDs sin ningún valor (nulos reales): 1379
  % nulos reales (sobre IDs únicos):   15.2%


### Tratamiento de nulos en FVC

Como se ha calculado arriba, el **15.2% de los pacientes únicos** (1.379 pacientes)
no tienen ningún valor registrado en ninguna de sus visitas — estos son los *nulos reales*
que requieren imputación con la mediana.

El resto de nulos (a nivel de fila) se resuelven propagando valores entre visitas
del mismo paciente con `ffill`/`bfill`. La estrategia completa es:

1. **Ordenar** por `Forced_Vital_Capacity_Delta` para garantizar orden temporal
2. **Propagar** con `ffill`/`bfill` dentro de cada paciente — así un valor de visita 2
   rellena las filas vacías de otras visitas del mismo paciente
3. **Extraer el basal**: primera fila por paciente tras la propagación
4. **Imputar con mediana** únicamente los 1.379 pacientes que siguen sin ningún valor

Esto evita imputar artificialmente a pacientes que sí tienen medición en alguna visita.

In [15]:
# 1. Ordenar por delta temporal
fvc_sorted = fvc_clean.sort_values(['subject_id', 'Forced_Vital_Capacity_Delta'])

# 2. Propagar valores dentro de cada paciente
fvc_sorted['Subject_Liters_Trial_1'] = (
    fvc_sorted
    .groupby('subject_id')['Subject_Liters_Trial_1']
    .transform(lambda g: g.ffill().bfill())
)

# 3. Diagnóstico de nulos reales tras propagación
nulos_reales = (
    fvc_sorted.groupby('subject_id')['Subject_Liters_Trial_1']
    .apply(lambda g: g.isna().all())
    .sum()
)
print(f"Nulos reales tras propagación: {nulos_reales}")

# 4. Quedarse con valor basal usando nth(0)
fvc_basal = (
    fvc_sorted
    .groupby('subject_id')
    .nth(0)
    .reset_index()
    [['subject_id', 'Subject_Liters_Trial_1']]
    .rename(columns={'Subject_Liters_Trial_1': 'fvc_basal'})
)

# 5. Imputar con mediana solo los nulos reales
mediana_fvc = fvc_basal['fvc_basal'].median()
fvc_basal['fvc_basal'] = fvc_basal['fvc_basal'].fillna(mediana_fvc)

print(f"Mediana FVC usada: {mediana_fvc:.3f} L")
print(f"Shape final: {fvc_basal.shape}")

Nulos reales tras propagación: 1379
Mediana FVC usada: 3.090 L
Shape final: (9090, 2)


## 6. Extracción de variables de Vitals

De los signos vitales extraemos el **peso basal** del paciente. El paper de 
Atassi et al. (2014) identificó el IMC como predictor de progresión más lenta;
pacientes con mayor masa corporal tienden a progresar más despacio.

El resto de columnas se descartan por tener más del 80% de nulos o por no 
tener relevancia clínica directa para la progresión de ELA.

In [16]:
# Selección de columnas relevantes
vitals_clean = vitals[['subject_id', 'Vital_Signs_Delta', 
                         'Weight', 'Weight_Units']].copy()

print("Shape Vitals seleccionado:", vitals_clean.shape)
print("\nIDs únicos:", vitals_clean['subject_id'].nunique())
print("\nUnidades de peso disponibles:")
print(vitals_clean['Weight_Units'].value_counts())

Shape Vitals seleccionado: (89065, 4)

IDs únicos: 11686

Unidades de peso disponibles:
Weight_Units
Kilograms    51562
Pounds         385
Name: count, dtype: int64


### Tratamiento de nulos en Vitals

`Weight` tiene un 29.9% de nulos. Aplicamos **imputación con la mediana**
para conservar todos los pacientes.

Antes de imputar verificamos las unidades; si hay pesos en libras (Pounds) 
los convertimos a kilogramos para homogeneizar.
Tomamos la primera medición de cada paciente asociada a un valor como medida basal.

In [17]:
# 1. Convertir unidades
vitals_clean['Weight'] = vitals_clean.apply(
    lambda row: row['Weight'] * 0.453592 
    if row['Weight_Units'] == 'Pounds' else row['Weight'], axis=1
)

# 2. Ordenar por delta temporal
vitals_sorted = vitals_clean.sort_values(['subject_id', 'Vital_Signs_Delta'])

# 3. Propagar valores dentro de cada paciente antes de coger el primero
vitals_sorted['Weight'] = (
    vitals_sorted
    .groupby('subject_id')['Weight']
    .transform(lambda g: g.ffill().bfill())
)

# 4. DIAGNÓSTICO de nulos reales (tras propagar dentro de cada paciente)
nulos_reales = (
    vitals_sorted.groupby('subject_id')['Weight']
    .apply(lambda g: g.isna().all())
    .sum()
)
print(f"Pacientes sin ningún peso registrado (nulos reales): {nulos_reales}")
print(f"Pacientes con al menos un peso: {vitals_sorted['subject_id'].nunique() - nulos_reales}")

# 5. Quedarse con el primer registro
vitals_basal = vitals_sorted.groupby('subject_id').nth(0).reset_index()
vitals_basal = vitals_basal[['subject_id', 'Weight']].rename(
    columns={'Weight': 'weight_basal'}
)
# 6. Imputar con mediana solo los nulos reales
mediana_weight = vitals_basal['weight_basal'].median()
vitals_basal['weight_basal'] = vitals_basal['weight_basal'].fillna(mediana_weight)

print(f"Mediana peso usada: {mediana_weight:.1f} kg")

print("Shape final:", vitals_basal.shape)

Pacientes sin ningún peso registrado (nulos reales): 1806
Pacientes con al menos un peso: 9880
Mediana peso usada: 73.0 kg
Shape final: (11686, 2)


## 7. Extracción de variables de Labs

El archivo Labs está en **formato largo** (cada fila representa un análisis 
distinto de un paciente). Para usarlo en el modelo necesitamos convertirlo a 
**formato ancho** (una columna por cada análisis) mediante un pivotado.

Extraemos 4 biomarcadores con relevancia clínica en ELA respaldada por 
Atassi et al. (2014):
- `Creatinine` → marcador de masa muscular
- `Uric Acid` → marcador de estrés oxidativo
- `Creatine Kinase` → marcador de daño muscular
- `Albumin` → marcador de estado nutricional


In [18]:
# Filtrar solo los análisis que nos interesan
labs_filtrado = labs[labs['Test_Name'].isin([
    'Creatinine', 'Uric Acid', 'Creatine Kinase', 'Albumin'
])].copy()

print("Shape Labs filtrado:", labs_filtrado.shape)
print("\nRegistros por análisis:")
print(labs_filtrado['Test_Name'].value_counts())

Shape Labs filtrado: (222545, 5)

Registros por análisis:
Test_Name
Creatinine         73885
Albumin            65494
Creatine Kinase    56804
Uric Acid          26362
Name: count, dtype: int64


### Tratamiento de nulos en Labs

Tomamos la primera medición de cada paciente para cada análisis (valor basal).
Posteriormente pivotamos el dataframe de formato largo a formato ancho.

Estrategia de tratamiento de nulos:
- `creatinine` (1.2% nulos) → **eliminación de filas** por ser un porcentaje mínimo
- `albumin` (23.7% nulos) → **imputación con mediana**
- `creatine_kinase` (26.2% nulos) → **imputación con mediana**
- `uric_acid` (58.4% nulos) → **imputación con mediana** + creación de columna 
  binaria `uric_acid_unknown` para preservar la información de ausencia, 
  siguiendo el mismo criterio aplicado con `Site_of_Onset`

In [19]:
# Quedarnos con la primera medición de cada paciente y análisis
labs_sorted = labs_filtrado.sort_values(['subject_id', 'Laboratory_Delta'])
labs_basal  = labs_sorted.groupby(['subject_id', 'Test_Name']).first().reset_index()

# Pivotar de formato largo a formato ancho
labs_pivot = labs_basal.pivot(
    index='subject_id', 
    columns='Test_Name', 
    values='Test_Result'
).reset_index()

# Renombrar columnas
labs_pivot.columns.name = None
labs_pivot = labs_pivot.rename(columns={
    'Creatinine'    : 'creatinine',
    'Uric Acid'     : 'uric_acid',
    'Creatine Kinase': 'creatine_kinase',
    'Albumin'       : 'albumin'
})

print("Shape Labs pivotado:", labs_pivot.shape)
print("\nIDs únicos:", labs_pivot['subject_id'].nunique())

print("\nNulos por columna:")
print(labs_pivot.isnull().sum())
labs_pivot.head()

Shape Labs pivotado: (10006, 5)

IDs únicos: 10006

Nulos por columna:
subject_id            0
albumin            2377
creatine_kinase    2625
creatinine          117
uric_acid          5851
dtype: int64


,subject_id,albumin,creatine_kinase,creatinine,uric_acid
0,204,NaN,NaN,70.72,NaN
1,1904,NaN,545,61,481
2,3301,NaN,NaN,106.08,279.556
3,3650,42,85,61.88,NaN
4,3752,42,141,61.88,172.492


### Imputación de nulos y limpieza final de Labs

Aplicamos la estrategia definida: eliminamos las filas con `creatinine` nulo (1.2%), convertimos las columnas a numérico, creamos la variable indicadora `uric_acid_unknown` e imputamos con la mediana los valores ausentes de `albumin`, `creatine_kinase` y `uric_acid`.

In [20]:
# Creatinine — eliminar filas con nulos (solo 1.2%)
labs_pivot = labs_pivot.dropna(subset=['creatinine'])

# Convertir columnas a numérico
cols_numericas = ['albumin', 'creatine_kinase', 'uric_acid', 'creatinine']
for col in cols_numericas:
    labs_pivot[col] = pd.to_numeric(labs_pivot[col], errors='coerce')

# Uric acid — crear columna unknown antes de imputar
labs_pivot['uric_acid_unknown'] = labs_pivot['uric_acid'].isnull().astype(int)

# Imputar albumin, creatine_kinase y uric_acid con mediana
cols_imputar = ['albumin', 'creatine_kinase', 'uric_acid']

for col in cols_imputar:
    mediana = labs_pivot[col].median()
    labs_pivot[col] = labs_pivot[col].fillna(mediana)
    print(f"{col} → mediana: {mediana:.2f}")

print("\n✅ Nulos tras tratamiento:")
print(labs_pivot.isnull().sum())
print("\nShape final:", labs_pivot.shape)
print(f"\nPacientes sin medición de ácido úrico: {labs_pivot['uric_acid_unknown'].sum()}")

print("\n✅ Nulos finales:")
print(labs_pivot.isnull().sum())
print("\nShape final Labs:", labs_pivot.shape)

albumin → mediana: 44.00
creatine_kinase → mediana: 202.00
uric_acid → mediana: 297.40

✅ Nulos tras tratamiento:
subject_id           0
albumin              0
creatine_kinase      0
creatinine           3
uric_acid            0
uric_acid_unknown    0
dtype: int64

Shape final: (9889, 6)

Pacientes sin medición de ácido úrico: 5735

✅ Nulos finales:
subject_id           0
albumin              0
creatine_kinase      0
creatinine           3
uric_acid            0
uric_acid_unknown    0
dtype: int64

Shape final Labs: (9889, 6)


## 8. Unión de todos los dataframes

Unimos todos los dataframes procesados en un único dataset final usando 
`subject_id` como clave. Usamos `inner join` — solo conservamos pacientes 
que tengan datos en todos los archivos.

In [21]:
# Unir todos los dataframes
df_final = df_tasas.merge(demo_clean,    on='subject_id', how='inner')
df_final = df_final.merge(alsfrs_basal,  on='subject_id', how='inner')
df_final = df_final.merge(historia_clean,on='subject_id', how='inner')
df_final = df_final.merge(fvc_basal,     on='subject_id', how='inner')
df_final = df_final.merge(vitals_basal,  on='subject_id', how='inner')
df_final = df_final.merge(labs_pivot,    on='subject_id', how='inner')

print("Shape df_final:", df_final.shape)
print("\nColumnas:")
print(df_final.columns.tolist())
print("\nNulos por columna:")
print(df_final.isnull().sum())

Shape df_final: (2219, 24)

Columnas:
['subject_id', 'tasa_progresion', 'progresion_rapida', 'etiqueta', 'Age', 'Sex', 'alsfrs_basal_total', 'basal_bulbar', 'basal_motor_fino', 'basal_motor_grueso', 'basal_respiratorio', 'onset_limb', 'onset_bulbar', 'onset_spine', 'onset_limb_bulbar', 'onset_other', 'onset_unknown', 'fvc_basal', 'weight_basal', 'albumin', 'creatine_kinase', 'creatinine', 'uric_acid', 'uric_acid_unknown']

Nulos por columna:
subject_id            0
tasa_progresion       0
progresion_rapida     0
etiqueta              0
Age                   0
Sex                   0
alsfrs_basal_total    0
basal_bulbar          0
basal_motor_fino      0
basal_motor_grueso    0
basal_respiratorio    0
onset_limb            0
onset_bulbar          0
onset_spine           0
onset_limb_bulbar     0
onset_other           0
onset_unknown         0
fvc_basal             0
weight_basal          0
albumin               0
creatine_kinase       0
creatinine            0
uric_acid             0
ur

## 9. Limpieza final y exportación

Eliminamos columnas que no usaremos en el modelo y exportamos:
- `datos_modelos.csv` → solo variables numéricas para entrenar los modelos

In [22]:
# Eliminar columnas no necesarias para el modelo
df_final = df_final.drop(columns=['tasa_progresion'])

# Verificar distribución final de la variable objetivo
n_total  = len(df_final)
n_rapida = df_final['progresion_rapida'].sum()
n_lenta  = n_total - n_rapida

print(f"Total pacientes en dataset final: {n_total}")
print(f"Progresión rápida (1): {n_rapida} ({n_rapida/n_total*100:.1f}%)")
print(f"Progresión lenta  (0): {n_lenta} ({n_lenta/n_total*100:.1f}%)")
print(f"\nNúmero de variables predictoras: {df_final.shape[1] - 3}")

Total pacientes en dataset final: 2219
Progresión rápida (1): 971 (43.8%)
Progresión lenta  (0): 1248 (56.2%)

Número de variables predictoras: 20


### Exportación de datasets

Se genera la versión del dataset final:
- `datos_modelos.csv` → separador coma, punto decimal — para Python/sklearn

In [23]:
# Exportar para modelos (sin etiqueta) — punto decimal estándar para Python/sklearn
cols_modelos = [c for c in df_final.columns if c != 'etiqueta']
df_final[cols_modelos].to_csv(ruta_out + "datos_modelos.csv", index=False)


print("✅ Archivo exportado:")
print(f"- datos_modelos.csv → punto decimal (para Python)")
print(f"\nRuta: {ruta_out}")

✅ Archivo exportado:
- datos_modelos.csv → punto decimal (para Python)

Ruta: C:\Users\Paula\SAD_ELA\datos_procesados/
